# EdgeMem — Kaggle GPU Baseline Notebook

**Purpose**: Run the EdgeMem 20-task Pass@1 benchmark against a local open-weight model on a free Kaggle T4/P100 GPU. Produces a JSON scorecard and Markdown report in `/kaggle/working/`.

> **Hardware target**: Free Kaggle T4 (16 GB VRAM) or P100 (16 GB VRAM).  
> **Caveat (from `docs/03_IMPLEMENTATION_PLAN.md`)**: GPU-measured TTFT/tok/s/RSS numbers from this notebook are **accuracy-only reference data**, not hardware telemetry for the T1 CPU target. Re-measure RSS and speculative-decoding gains on real target-class hardware before publishing.

## Cell 1 — Configuration (edit here)

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────
# Change these values to switch models or endpoints.

MODEL_NAME   = "qwen2.5-coder:3b-instruct-q4_K_M"  # Ollama model tag
# Alternatives (Apache-2.0 licensed, fit in 16 GB VRAM):
#   "qwen2.5-coder:1.5b-instruct-q4_K_M"
#   "qwen2.5:3b-instruct-q4_K_M"
#   "starcoder2:3b-q4_K_M"

OLLAMA_HOST  = "http://127.0.0.1:11434"
BASE_URL     = f"{OLLAMA_HOST}/v1"
LABEL        = "kaggle-gpu-baseline"  # used in output filenames
OUTPUT_DIR   = "/kaggle/working/bench_results"
BENCH_REPO   = "https://github.com/YOUR_USERNAME/edgemem.git"  # set your fork URL
# If you don't have a public fork, the cells below use an inline copy instead.
USE_GIT_CLONE = False  # set True to clone from BENCH_REPO

# Optional: extra Ollama parameters
EXTRA_BODY = {"options": {"num_ctx": 8192, "temperature": 0.0}}

print(f"Model:     {MODEL_NAME}")
print(f"Base URL:  {BASE_URL}")
print(f"Label:     {LABEL}")
print(f"Output:    {OUTPUT_DIR}")

## Cell 2 — Install Ollama and start GPU daemon

In [ ]:
import subprocess, os, time, urllib.request, json, sys, pathlib

def run(cmd, **kw):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    if result.stdout: print(result.stdout[:2000])
    if result.stderr: print(result.stderr[:1000], file=sys.stderr)
    return result

# Install Ollama (Linux one-liner)
run("curl -fsSL https://ollama.com/install.sh | sh")

# Start Ollama server in background
proc = subprocess.Popen(
    ["ollama", "serve"],
    env={**os.environ, "OLLAMA_HOST": OLLAMA_HOST},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f"Ollama PID: {proc.pid}")

# Wait for Ollama to be ready
for i in range(30):
    try:
        with urllib.request.urlopen(f"{OLLAMA_HOST}/api/tags", timeout=3) as r:
            if r.status == 200:
                print(f"Ollama ready after {i+1}s")
                break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama did not start within 30s")

print("✅ Ollama running")

## Cell 3 — Pull model

In [ ]:
print(f"Pulling {MODEL_NAME} ...")
result = subprocess.run(
    ["ollama", "pull", MODEL_NAME],
    text=True, capture_output=True,
    env={**os.environ, "OLLAMA_HOST": OLLAMA_HOST}
)
print(result.stdout[-1000:] if result.stdout else "(no stdout)")
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)
    raise RuntimeError(f"ollama pull failed: {result.returncode}")
print(f"✅ {MODEL_NAME} pulled")

## Cell 4 — Load benchmark assets
Either clones from GitHub or uses an embedded copy of `bench_tasks.json`.

In [ ]:
WORK = pathlib.Path("/kaggle/working")
BENCH_DIR = WORK / "edgemem_bench"
BENCH_DIR.mkdir(parents=True, exist_ok=True)

if USE_GIT_CLONE:
    run(f"git clone --depth 1 {BENCH_REPO} {BENCH_DIR}")
    TASKS_FILE   = BENCH_DIR / "bench" / "bench_tasks.json"
    RUNNER_FILE  = BENCH_DIR / "bench" / "run_bench.py"
else:
    # ── Inline assets (populated by build script) ──────────────────────────
    # bench_tasks.json is embedded as a Python string below so the notebook
    # is fully self-contained for Kaggle upload.
    TASKS_FILE  = BENCH_DIR / "bench_tasks.json"
    RUNNER_FILE = BENCH_DIR / "run_bench.py"

    # -- bench_tasks.json: download from raw GitHub or paste inline --
    # For a self-contained notebook, copy bench_tasks.json content here:
    RAW_TASKS_URL = (
        "https://raw.githubusercontent.com/YOUR_USERNAME/edgemem/main/bench/bench_tasks.json"
    )
    RAW_RUNNER_URL = (
        "https://raw.githubusercontent.com/YOUR_USERNAME/edgemem/main/bench/run_bench.py"
    )
    try:
        for url, dest in [(RAW_TASKS_URL, TASKS_FILE), (RAW_RUNNER_URL, RUNNER_FILE)]:
            with urllib.request.urlopen(url, timeout=20) as r:
                dest.write_bytes(r.read())
        print("✅ Fetched bench assets from GitHub")
    except Exception as exc:
        print(f"⚠ Could not fetch from GitHub ({exc}). Paste bench_tasks.json into BENCH_DIR manually.")
        # Fallback: check if Kaggle Dataset is attached
        kaggle_dataset_tasks = pathlib.Path("/kaggle/input/edgemem-bench/bench_tasks.json")
        kaggle_dataset_runner = pathlib.Path("/kaggle/input/edgemem-bench/run_bench.py")
        if kaggle_dataset_tasks.exists():
            import shutil
            shutil.copy(kaggle_dataset_tasks, TASKS_FILE)
            shutil.copy(kaggle_dataset_runner, RUNNER_FILE)
            print("✅ Loaded bench assets from Kaggle Dataset")
        else:
            raise FileNotFoundError(
                "bench_tasks.json not found. Set USE_GIT_CLONE=True or attach a Kaggle Dataset."
            )

assert TASKS_FILE.exists(),  f"Missing: {TASKS_FILE}"
assert RUNNER_FILE.exists(), f"Missing: {RUNNER_FILE}"

tasks = json.loads(TASKS_FILE.read_text())
print(f"Loaded {len(tasks)} tasks from {TASKS_FILE}")
from collections import Counter
cats = Counter(t['category'] for t in tasks)
print(f"Categories: {dict(cats)}")

## Cell 5 — Run full 20-task Pass@1 benchmark

In [ ]:
import json, os

output_path = pathlib.Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

extra_body_str = json.dumps(EXTRA_BODY)

cmd = [
    sys.executable, str(RUNNER_FILE),
    "--base-url",        BASE_URL,
    "--model",           MODEL_NAME,
    "--api-key-env",     "",          # no API key for local Ollama
    "--tasks-file",      str(TASKS_FILE),
    "--label",           LABEL,
    "--output-dir",      str(output_path),
    "--extra-body-json", extra_body_str,
    "--timeout-llm",     "180",
    "--timeout-task",    "15",
]

print("Running benchmark...")
print(" ".join(cmd))
print()

t0 = time.time()
proc = subprocess.run(cmd, text=True, capture_output=False)  # live output
elapsed = time.time() - t0
print(f"\nBenchmark finished in {elapsed:.1f}s  (exit code {proc.returncode})")

## Cell 6 — Display results

In [ ]:
results_json = output_path / f"{LABEL}_results.json"
results_md   = output_path / f"{LABEL}_report.md"

if results_json.exists():
    data = json.loads(results_json.read_text())
    p = data['metrics']['pass_at_1']
    l = data['metrics']['latency']
    t = data['metrics']['tokens']
    c = p['counts']
    print("="*60)
    print(f"  Pass@1 (overall):    {p['overall']:.1%}  ({c['passed']}/{c['total']})")
    if p.get('exact_fact') is not None:
        print(f"  Pass@1 exact_fact:   {p['exact_fact']:.1%}  ({c['exact_fact_passed']}/{c['exact_fact_total']})")
    if p.get('continuity') is not None:
        print(f"  Pass@1 continuity:   {p['continuity']:.1%}  ({c['continuity_passed']}/{c['continuity_total']})")
    print(f"  Total prompt tokens:  {t['total_prompt']:,}")
    print(f"  Total comp tokens:    {t['total_completion']:,}")
    if l.get('mean_ttft_ms') is not None:
        print(f"  Mean TTFT (ms):      {l['mean_ttft_ms']}  p95={l['p95_ttft_ms']}")
    if l.get('mean_gen_toks_s') is not None:
        print(f"  Mean gen tok/s:      {l['mean_gen_toks_s']}")
    print(f"  Total wall time (s): {l['total_wall_s']}")
    print("="*60)
    print()
    print("Per-task breakdown:")
    for r in data['results']:
        icon = '✅' if r['passed'] else '❌'
        err  = f"  → {r['error'][:80]}" if not r['passed'] and r.get('error') else ''
        print(f"  {icon} {r['task_id']:10s} [{r['category']}]{err}")
else:
    print("⚠ results JSON not found — check Cell 5 for errors")

if results_md.exists():
    print(f"\nMarkdown report saved to: {results_md}")
    # Also display inline
    try:
        from IPython.display import Markdown, display
        display(Markdown(results_md.read_text()))
    except ImportError:
        print(results_md.read_text()[:3000])

## Cell 7 — Save outputs

Files in `/kaggle/working/` are automatically saved as Kaggle output.
List them here for reference.

In [ ]:
print("Output files:")
for f in sorted(output_path.rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.relative_to('/kaggle/working')}  ({size_kb:.1f} KB)")

print()
print("⚠  Reminder (from Implementation Plan Phase 1 caveat):")
print("   GPU-measured TTFT/tok/s/RSS from this Kaggle notebook are accuracy-only")
print("   reference data. Re-measure on real T1-class hardware (dual-core CPU, 8 GB")
print("   RAM) before publishing any performance claim.")